In [0]:
# %sql
# -- 1. Aseguramos que la tabla no tenga basura previa
# DROP TABLE IF EXISTS workspace.formula_1.bronze_drivers;

In [0]:
# %sql
# SELECT * FROM read_files('/Volumes/workspace/formula_1/formula_1/drivers', multiLine => true) LIMIT 5

In [0]:
%sql
-- 1. Crear la tabla SOLO si no existe
-- Usamos TBLPROPERTIES para soportar nombres de columnas con espacios (como tus 'Opcion 1')
CREATE TABLE IF NOT EXISTS workspace.formula_1.bronze_drivers
USING DELTA
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT get_json_object(raw_json, '$.full_name')) AS total_drivers,
    COUNT(DISTINCT get_json_object(raw_json, '$.team_name')) AS total_teams 
FROM 
    workspace.formula_1.bronze_drivers

In [0]:
%sql
COPY INTO workspace.formula_1.bronze_drivers
FROM (
  SELECT 
    to_json(struct(*)) AS raw_json,
    month,
    year,
    _metadata.file_modification_time AS file_metadata_modification_time,
    _metadata.file_path AS file_metadata_path
  FROM '/Volumes/workspace/formula_1/formula_1/drivers'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true', 'force' = 'false');

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows_after_copy_into,
    COUNT(DISTINCT get_json_object(raw_json, '$.full_name')) AS total_drivers_after_copy_into,
    COUNT(DISTINCT get_json_object(raw_json, '$.team_name')) AS total_teams_after_copy_into 
FROM 
    workspace.formula_1.bronze_drivers

In [0]:
%sql
SELECT 
* 
FROM 
workspace.formula_1.bronze_drivers
ORDER BY
file_metadata_modification_time DESC
LIMIT 5;